# Exploration de Geolocation — phase 1

Lecture seule sur C:, .venv du projet. Exécuter de haut en bas depuis la racine ou notebooks/.
Grain observé : enregistrement géographique, sans clé unique présumée. Un préfixe peut avoir plusieurs points.
Le notebook conserve un seul tableau principal ; villes/États en category, coordonnées source en string.
Aucun CSV exporté, aucun validateur créé. Rapport : docs/GEOLOCATION_EXPLORATION.md.
Code exécuté avec Python le 2026-10-24 ; notebook livré sans sorties sauvegardées ni lancement de noyau Jupyter.

In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import re
import sys
import unicodedata

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "AGENTS.md").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "AGENTS.md").is_file()
paths = {name: PROJECT_ROOT / "data" / filename for name,filename in {
    "geolocation": "olist_geolocation_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "customers": "olist_customers_dataset.csv",
}.items()}

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

source_hashes = {name: sha256(path) for name,path in paths.items()}
geo = pd.read_csv(paths["geolocation"], keep_default_na=False, dtype={
    "geolocation_zip_code_prefix": "string", "geolocation_lat": "string",
    "geolocation_lng": "string", "geolocation_city": "category",
    "geolocation_state": "category",
})
sample = pd.read_csv(paths["geolocation"], nrows=1000)
PREFIX, LAT, LNG, CITY, STATE = geo.columns
report = {
    "environment": {"python": platform.python_version(), "pandas": pd.__version__,
                    "executable": sys.executable},
    "shape": list(geo.shape),
    "source_size_bytes": paths["geolocation"].stat().st_size,
    "source_sha256": source_hashes,
    "inferred_dtypes_sample_1000": {col:str(dtype) for col,dtype in sample.dtypes.items()},
    "analysis_dtypes": {col:str(dtype) for col,dtype in geo.dtypes.items()},
    "main_dataframe_deep_MiB": round(float(geo.memory_usage(deep=True).sum())/2**20, 3),
    "parser_nulls": {col:int(value) for col,value in geo.isna().sum().items()},
    "empty": {}, "whitespace_only_nonempty": {}, "outer_whitespace": {},
    "exact_duplicate_excess": int(geo.duplicated().sum()),
    "exact_duplicate_involved_rows": int(geo.duplicated(keep=False).sum()),
}
for col in geo:
    text = geo[col].astype("string")
    report["empty"][col] = int(text.eq("").sum())
    report["whitespace_only_nonempty"][col] = int((text.ne("") & text.str.strip().eq("")).sum())
    report["outer_whitespace"][col] = int(text.ne(text.str.strip()).sum())
report["distinct_complete_rows"] = len(geo) - report["exact_duplicate_excess"]

def counts(series):
    return {str(key): int(value) for key,value in series.value_counts().items()}

def records(frame):
    return json.loads(frame.to_json(orient="records", force_ascii=True))
report

## Préfixes et coordonnées

La lecture textuelle préserve les zéros et les représentations sources pour les doublons complets.
Les types inférés sur 1 000 lignes sont indicatifs ; les contrôles numériques ci-dessous couvrent toutes les lignes.
Bornes mondiales : latitude [-90,90], longitude [-180,180], bornes incluses.
Une coordonnée positive, extrême ou éloignée du reste n'est pas automatiquement impossible ; aucun polygone officiel du Brésil n'est utilisé.

In [ ]:
prefix = geo[PREFIX]
report["postal"] = {
    "unique": int(prefix.nunique()),
    "length_distribution": counts(prefix.str.len()),
    "invalid_five_ascii_digits": int((~prefix.str.fullmatch("[0-9]{5}", na=False)).sum()),
    "leading_zero_rows": int(prefix.str.startswith("0", na=False).sum()),
    "leading_zero_distinct_prefixes": int(prefix[prefix.str.startswith("0", na=False)].nunique()),
    "all_zero_rows": int(prefix.eq("00000").sum()),
    "min": prefix.min(), "max": prefix.max(),
}
numeric = geo[[LAT,LNG]].apply(pd.to_numeric, errors="coerce")
report["coordinates"] = {}
valid_pairs = pd.Series(True, index=geo.index)
for col, lower, upper in [(LAT,-90,90), (LNG,-180,180)]:
    text = geo[col].str.strip()
    missing = text.eq("") | geo[col].isna()
    values = numeric[col]
    finite = (values.notna() & np.isfinite(values)).fillna(False)
    within = values.between(lower,upper)
    report["coordinates"][col] = {
        "numeric_dtype_full_column": str(values.dtype),
        "missing_or_blank": int(missing.sum()),
        "nonconvertible_present": int((~missing & values.isna()).sum()),
        "nonfinite_present": int((values.notna() & ~np.isfinite(values)).sum()),
        "outside_global_bounds_finite": int((finite & ~within).sum()),
        "zero_rows": int(values.eq(0).sum()),
        "positive_rows": int(values.gt(0).sum()),
        "statistics_finite": values[finite].describe(percentiles=[.01,.25,.5,.75,.99]).to_dict(),
    }
    valid_pairs &= finite & within
report["invalid_coordinate_pair_rows"] = int((~valid_pairs).sum())
report["invalid_coordinate_examples"] = records(geo.loc[~valid_pairs].head(10))
report["postal"], report["coordinates"]

## Villes, États et répétitions par préfixe

Clé exploratoire : minuscules casefold, NFKD, retrait des accents, ponctuation en espaces, espaces réduits.
Aucune correction des libellés. Comparaison des variantes à État identique.
Coordonnées distinctes calculées après conversion numérique ; aucun arrondi imposé.

In [ ]:
VALID_STATES = {
    "AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG",
    "PA","PB","PR","PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO",
}
def city_key(value):
    if pd.isna(value):
        return None
    text = unicodedata.normalize("NFKD", str(value).casefold())
    text = "".join(char for char in text if not unicodedata.combining(char))
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9]+", " ", text)).strip()

city_keys = {city: city_key(city) for city in geo[CITY].cat.categories}
unique_names = geo[[STATE,CITY]].drop_duplicates().copy()
unique_names["city_key"] = unique_names[CITY].astype("string").map(city_keys)
variants = unique_names.groupby([STATE,"city_key"], observed=True)[CITY].nunique()
variant_keys = variants[variants.gt(1)].index
variant_examples = []
for state,key in list(variant_keys)[:10]:
    names = unique_names.loc[(unique_names[STATE]==state) & (unique_names.city_key==key), CITY]
    variant_examples.append({"state":str(state), "city_key":key, "spellings":names.astype("string").tolist()})
report["cities_states"] = {
    "unique_cities": int(geo[CITY].nunique()),
    "normalized_city_keys": len(set(city_keys.values())),
    "variant_groups_same_state": len(variant_keys),
    "variant_examples": variant_examples,
    "state_distribution": counts(geo[STATE]),
    "unknown_state_rows": int((~geo[STATE].isin(VALID_STATES)).sum()),
    "unknown_state_values": sorted(set(geo[STATE].astype("string")) - VALID_STATES),
}
grouped = geo.groupby(PREFIX, observed=True)
prefix_counts = grouped.size()
city_counts = grouped[CITY].nunique()
state_counts = grouped[STATE].nunique()
pair_frame = pd.DataFrame({PREFIX:prefix, LAT:numeric[LAT], LNG:numeric[LNG]})
coordinate_counts = pair_frame.loc[valid_pairs].drop_duplicates().groupby(PREFIX).size()
report["prefix_multiplicity"] = {
    "prefixes_multiple_rows": int(prefix_counts.gt(1).sum()),
    "rows_per_prefix_statistics": prefix_counts.describe(percentiles=[.5,.95,.99]).to_dict(),
    "prefixes_multiple_valid_coordinate_pairs": int(coordinate_counts.gt(1).sum()),
    "distinct_valid_prefix_coordinate_triples": int(coordinate_counts.sum()),
    "prefixes_multiple_raw_cities": int(city_counts.gt(1).sum()),
    "prefixes_multiple_states": int(state_counts.gt(1).sum()),
    "multi_state_prefix_examples": records(
        geo.loc[prefix.isin(state_counts[state_counts.gt(1)].index), [PREFIX,CITY,STATE]].drop_duplicates().head(20)),
}
del pair_frame
report["cities_states"], report["prefix_multiplicity"]

## Dispersion et extrêmes descriptifs

Étendue = maximum - minimum, par axe et par préfixe, sur les seules paires mondialement valides.
Unité : degrés, sans assimilation à des kilomètres ni à un diamètre réel.
Les coordonnées extrêmes sont montrées sans rejet automatique ; aucun seuil de dispersion arbitraire.

In [ ]:
valid_numeric = numeric.loc[valid_pairs].assign(prefix=prefix[valid_pairs])
bounds = valid_numeric.groupby("prefix")[[LAT,LNG]].agg(["min","max"])
spans = pd.DataFrame({
    "latitude_span_degrees": bounds[(LAT,"max")] - bounds[(LAT,"min")],
    "longitude_span_degrees": bounds[(LNG,"max")] - bounds[(LNG,"min")],
})
report["dispersion"] = {
    "method": "Axis-aligned max-min in degrees per prefix, globally valid numeric pairs only; no rejection threshold, not distances or true diameters.",
    "statistics": spans.describe(percentiles=[.5,.95,.99]).to_dict(),
    "top_latitude_spans": records(spans.nlargest(10,"latitude_span_degrees").reset_index()),
    "top_longitude_spans": records(spans.nlargest(10,"longitude_span_degrees").reset_index()),
}
report["coordinate_extremes"] = {
    col: {"lowest": records(geo.loc[numeric[col].nsmallest(5).index]),
          "highest": records(geo.loc[numeric[col].nlargest(5).index])}
    for col in [LAT,LNG]
}
del valid_numeric, bounds
report["dispersion"], report["coordinate_extremes"]

## Couverture Sellers / Customers

Appartenance aux ensembles de préfixes, villes normalisées et États observés, sans jointure brute.
Les deux tables ne contiennent pas de latitude/longitude à comparer directement.
Indicateurs de coordonnées par préfixe : présence de points invalides ou de plusieurs points valides, sans attribuer ces points aux clients/vendeurs individuellement.
Geolocation n'est ni un référentiel officiel ni exhaustif ; les écarts peuvent venir de toute table.

In [ ]:
prefix_set = set(prefix)
state_sets = grouped[STATE].agg(lambda values: set(values.dropna().astype(str))).to_dict()
city_sets = grouped[CITY].agg(lambda values: {city_keys[value] for value in values.dropna().unique()}).to_dict()
invalid_prefixes = set(prefix[~valid_pairs])
valid_prefixes = set(prefix[valid_pairs])
multi_coordinate_prefixes = set(coordinate_counts[coordinate_counts.gt(1)].index)
report["coverage"] = {}
for name in ["sellers","customers"]:
    stem = "seller" if name=="sellers" else "customer"
    cols = [f"{stem}_zip_code_prefix",f"{stem}_city",f"{stem}_state"]
    reference = pd.read_csv(paths[name], usecols=cols, dtype="string")
    codes = reference[cols[0]]
    known = codes.isin(prefix_set)
    state_diff = city_diff = 0
    for code,city,state in reference[cols].itertuples(index=False, name=None):
        if code not in prefix_set:
            continue
        state_diff += state not in state_sets[code]
        city_diff += city_key(city) not in city_sets[code]
    report["coverage"][name] = {
        "rows": len(reference),
        "distinct_prefixes": int(codes.nunique()),
        "covered_rows": int(known.sum()),
        "absent_prefix_rows": int((~known).sum()),
        "absent_distinct_prefixes": int(codes[~known].nunique()),
        "absent_prefix_examples": codes[~known].drop_duplicates().head(20).tolist(),
        "state_not_observed_for_known_prefix_rows": int(state_diff),
        "city_key_not_observed_for_known_prefix_rows": int(city_diff),
        "rows_on_prefixes_with_invalid_coordinate_pairs": int(codes.isin(invalid_prefixes).sum()),
        "rows_on_covered_prefixes_without_valid_coordinates": int((known & ~codes.isin(valid_prefixes)).sum()),
        "rows_on_prefixes_with_multiple_valid_coordinate_pairs": int(codes.isin(multi_coordinate_prefixes).sum()),
    }
assert source_hashes == {name:sha256(path) for name,path in paths.items()}
report["coverage"]

## Contrat proposé — attendre validation

Cinq colonnes obligatoires, préfixe renseigné de cinq chiffres ASCII, coordonnées renseignées/convertibles/finies dans les bornes mondiales inclusives, ville et État renseignés, État dans les 27 codes existants.
Pas d'unicité imposée au préfixe, pas de rejet des doublons complets ni des villes variantes sans politique validée.
Doublons, multi-villes/multi-États, dispersion et divergences inter-tables restent des indicateurs métier.
Ne pas choisir une coordonnée représentative (moyenne/médiane/premier point) avant validation d'une stratégie.
Aucune région arbitraire du Brésil ni seuil de distance ne sert de règle de rejet.

In [ ]:
print(json.dumps(report, ensure_ascii=True, indent=2))